In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from google.colab import files

# Step 1: Upload the Excel Files
print("Please upload your Excel files: positive.xlsx, negative.xlsx, neutral.xlsx")
uploaded = files.upload()  # This will prompt the user to upload files

# Load the datasets after upload
negative_data = pd.read_excel('negative.xlsx', names=['text'])
positive_data = pd.read_excel('positive.xlsx', names=['text'])
neutral_data = pd.read_excel('neutral.xlsx', names=['text'])

# Add sentiment labels
negative_data['sentiment'] = 'negative'
positive_data['sentiment'] = 'positive'
neutral_data['sentiment'] = 'neutral'

# Combine all datasets
data = pd.concat([negative_data, positive_data, neutral_data], ignore_index=True)

# Clean the text
def clean_text(text):
    if isinstance(text, str):  # Ensure the text is a string
        return text.replace('·', '').strip()
    return ''  # Replace invalid entries with an empty string

data['text'] = data['text'].apply(clean_text)

# Drop rows with empty or missing text
data = data[data['text'] != '']

# Map sentiments to numeric labels
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
data['sentiment'] = data['sentiment'].map(label_mapping)

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data['text'])
sequences = tokenizer.texts_to_sequences(data['text'])
max_sequence_length = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_sequence_length, padding='post')

# Prepare data for training
X = padded_sequences
y = data['sentiment'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build the RNN model
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 128

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_sequence_length),
    LSTM(128, return_sequences=True),
    Dropout(0.2),
    LSTM(64),
    Dropout(0.2),
    Dense(3, activation='softmax')  # 3 output classes: negative, neutral, positive
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1)

# Evaluate the model
y_pred = model.predict(X_test).argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'positive']))

# Test the model with one example
def predict_sentiment(text, tokenizer, model, max_sequence_length):
    # Clean and preprocess the input text
    text = text.replace('·', '').strip()
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence = pad_sequences(sequence, maxlen=max_sequence_length, padding='post')

    # Predict sentiment
    prediction = model.predict(padded_sequence).argmax(axis=1)[0]
    sentiment_mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}
    return sentiment_mapping[prediction]

# Example text in Kannada
example_text = "ನಾನು ಶಾಲೆಗೆ ಹೋಗುತ್ತಿದ್ದೇನೆ."
predicted_sentiment = predict_sentiment(example_text, tokenizer, model, max_sequence_length)

print(f"Text: {example_text}")
print(f"Predicted Sentiment: {predicted_sentiment}")

from sklearn.metrics import accuracy_score

# Evaluate the model
y_pred = model.predict(X_test).argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'positive']))

# Calculate accuracy in percentage
accuracy = accuracy_score(y_test, y_pred)
accuracy_percentage = accuracy * 100
print(f"Accuracy: {accuracy_percentage:.2f}%")

Please upload your Excel files: positive.xlsx, negative.xlsx, neutral.xlsx


Saving negative.xlsx to negative (5).xlsx
Saving neutral.xlsx to neutral (5).xlsx
Saving positive.xlsx to positive (5).xlsx
Epoch 1/10


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


39/39 ━━━━━━━━━━━━━━━━━━━━ 6s 47ms/step - accuracy: 0.4551 - loss: 1.0255 - val_accuracy: 0.6691 - val_loss: 0.7029
Epoch 2/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.8435 - loss: 0.3796 - val_accuracy: 0.8456 - val_loss: 0.4411
Epoch 3/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.9508 - loss: 0.1639 - val_accuracy: 0.7941 - val_loss: 0.6574
Epoch 4/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 4s 70ms/step - accuracy: 0.9759 - loss: 0.0752 - val_accuracy: 0.8676 - val_loss: 0.4433
Epoch 5/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.9967 - loss: 0.0110 - val_accuracy: 0.8382 - val_loss: 0.5784
Epoch 6/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.9951 - loss: 0.0105 - val_accuracy: 0.8382 - val_loss: 0.7955
Epoch 7/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.9985 - loss: 0.0091 - val_accuracy: 0.8382 - val_loss: 0.6494
Epoch 8/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 1.0000 - loss: 0.0013 - val_accuracy: 0.8456 - val_loss: 0.